# 검색 증강 생성 (Retrieval Augmented Generation)

검색 증강 생성(RAG)은 언어 모델에 외부 지식 소스를 활용할 수 있게 하는 기법입니다. 이 방식은 모델이 최신 정보나 특정 도메인 지식에 접근하여 더 정확하고 유용한 응답을 생성할 수 있게 합니다.

이 노트북에서는 RAG의 기본 개념을 배우고 간단한 RAG 시스템을 구현해보겠습니다.

In [24]:
# 필요한 라이브러리 및 모듈 임포트
import os
import sys
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# utils 디렉토리의 helpers.py 모듈을 임포트하기 위한 경로 설정
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.helpers import get_completion

# .env 파일 로드
load_dotenv()

True

## 1. RAG의 기본 개념

RAG는 크게 두 가지 주요 단계로 구성됩니다:

1. **검색(Retrieval)**: 사용자의 질문이나 입력을 기반으로 관련 정보를 외부 지식 소스(문서, 데이터베이스 등)에서 검색합니다.
2. **생성(Generation)**: 검색된 정보를 프롬프트에 통합하여 언어 모델에게 전달하고, 이를 기반으로 응답을 생성합니다.

RAG의 주요 장점은 다음과 같습니다:
- 모델의 훈련 데이터에 없는 최신 정보 접근 가능
- 사실적 정확도 향상
- 환각(hallucination) 감소
- 특정 도메인 지식에 대한 더 정확한 응답 제공

## 2. 간단한 RAG 시스템 구현하기

먼저 간단한 RAG 시스템을 구현해보겠습니다. 이를 위해 샘플 문서 데이터를 생성하고, 이를 검색하는 기능을 만들어보겠습니다.

In [35]:
# 샘플 문서 데이터 생성
sample_documents = [
    {
        "id": 1,
        "title": "파이썬 기초 프로그래밍",
        "content": "파이썬은 초보자부터 전문가까지 널리 사용되는 프로그래밍 언어입니다. 간결한 문법과 다양한 라이브러리로 인해 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 활용됩니다. 변수, 자료형, 조건문, 반복문 등 기본 개념을 이해하면 쉽게 배울 수 있습니다."
    },
    {
        "id": 2,
        "title": "머신러닝 입문",
        "content": "머신러닝은 컴퓨터가 데이터로부터 패턴을 학습하고 예측할 수 있게 하는 기술입니다. 지도학습, 비지도학습, 강화학습 등 다양한 학습 방식이 있으며, 분류, 회귀, 클러스터링 등의 작업에 활용됩니다. scikit-learn, TensorFlow, PyTorch 등의 라이브러리를 통해 구현할 수 있습니다."
    },
    {
        "id": 3,
        "title": "딥러닝의 이해",
        "content": "딥러닝은 인공신경망을 여러 층으로 쌓아 복잡한 패턴을 학습하는 머신러닝의 한 분야입니다. 이미지 인식, 자연어 처리, 음성 인식 등에서 뛰어난 성능을 보이며, CNN, RNN, Transformer 등 다양한 아키텍처가 있습니다. 대량의 데이터와 GPU를 활용한 학습이 일반적입니다."
    },
    {
        "id": 4,
        "title": "자연어 처리 기술",
        "content": "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 처리하는 기술입니다. 토큰화, 품사 태깅, 개체명 인식 등의 기본 작업부터 감정 분석, 기계 번역, 질의응답 시스템 등 고급 응용까지 다양합니다. BERT, GPT, T5 등의 트랜스포머 기반 모델이 현재 최고 성능을 보이고 있습니다."
    },
    {
        "id": 5,
        "title": "데이터 시각화 방법론",
        "content": "데이터 시각화는 데이터를 그래픽 요소로 표현하여 패턴, 추세, 이상점 등을 쉽게 파악할 수 있게 합니다. 막대 그래프, 선 그래프, 산점도, 히트맵 등 다양한 차트 유형이 있으며, Matplotlib, Seaborn, Plotly 등의 라이브러리를 통해 구현할 수 있습니다. 효과적인 시각화는 데이터 스토리텔링의 핵심입니다."
    },
        {
        "id": 6,
        "title": "인공신경망의 기초",
        "content": "인공신경망은 인간 뇌의 뉴런 구조에서 영감을 받은 컴퓨팅 모델입니다. 입력층, 은닉층, 출력층으로 구성되며 각 노드는 가중치와 활성화 함수를 통해 연결됩니다. 역전파 알고리즘을 통해 학습하며, 패턴 인식과 예측 작업에 탁월한 성능을 보입니다."
    },
    {
        "id": 7,
        "title": "컴퓨터 비전 개요",
        "content": "컴퓨터 비전은 기계가 디지털 이미지와 비디오를 분석하고 이해할 수 있게 하는 AI 분야입니다. 이미지 분류, 객체 감지, 이미지 분할, 얼굴 인식 등의 작업을 수행합니다. CNN(합성곱 신경망)이 주로 사용되며, 자율주행차, 의료영상 분석 등에 활용됩니다."
    },
    {
        "id": 8,
        "title": "강화학습 알고리즘",
        "content": "강화학습은 에이전트가 환경과 상호작용하며 보상을 최대화하는 방향으로 행동을 학습하는 기계학습 방법입니다. Q-러닝, 정책 경사법, 딥 Q-네트워크(DQN) 등의 알고리즘이 있으며, 게임 플레이, 로봇 제어, 자원 관리 등 다양한 문제에 적용됩니다."
    },
    {
        "id": 9,
        "title": "빅데이터 처리 기술",
        "content": "빅데이터 처리는 대용량 데이터를 효율적으로 저장, 처리, 분석하는 기술입니다. Hadoop, Spark, Kafka 등의 프레임워크가 활용되며, 분산 처리를 통해 대규모 데이터를 병렬로 처리합니다. 실시간 처리, 배치 처리 등 다양한 처리 방식이 있으며, 데이터 마이닝과 기계학습의 기반이 됩니다."
    },
    {
        "id": 10,
        "title": "자연어 이해(NLU) 기술",
        "content": "자연어 이해는 컴퓨터가 인간 언어의 의미를 파악하는 NLP의 하위 분야입니다. 의도 분류, 개체명 인식, 감정 분석 등을 통해 텍스트의 의미를 이해합니다. BERT, GPT 등의 트랜스포머 모델이 큰 발전을 이루었으며, 챗봇, 가상 비서, 검색 엔진 등에 적용됩니다."
    },
    {
        "id": 11,
        "title": "인공지능 윤리",
        "content": "인공지능 윤리는 AI 시스템 개발과 사용에 관련된 도덕적 문제를 다룹니다. 프라이버시, 편향성, 투명성, 책임성, 자율성 등이 주요 이슈이며, 인공지능의 발전에 따라 사회적, 법적 규제 프레임워크 개발이 중요해지고 있습니다. AI 기술이 공정하고 안전하게 사용되도록 하는 가이드라인이 필요합니다."
    },
    {
        "id": 12,
        "title": "지능형 데이터 분석",
        "content": "지능형 데이터 분석은 기계학습과 통계적 방법을 결합하여 데이터에서 유용한 인사이트를 추출하는 프로세스입니다. 탐색적 데이터 분석(EDA), 예측 분석, 처방적 분석 등의 단계를 포함하며, 비즈니스 의사결정, 과학적 연구, 사회 현상 분석 등 다양한 영역에서 활용됩니다."
    },
    {
        "id": 13,
        "title": "신경망 최적화 기법",
        "content": "신경망 최적화는 모델의 성능을 향상시키기 위한 다양한 기법을 포함합니다. 경사하강법, 모멘텀, Adam 등의 최적화 알고리즘과 배치 정규화, 드롭아웃, 조기 종료 등의 정규화 기법이 사용됩니다. 하이퍼파라미터 튜닝과 아키텍처 검색도 최적화의 중요한 부분입니다."
    },
    {
        "id": 14,
        "title": "추천 시스템 알고리즘",
        "content": "추천 시스템은 사용자의 선호도와 행동에 기반하여 개인화된 항목을 제안하는 AI 응용 프로그램입니다. 협업 필터링, 콘텐츠 기반 필터링, 하이브리드 방식 등의 접근법이 있으며, 행렬 분해, 딥러닝, 강화학습 등 다양한 기술이 활용됩니다. 전자상거래, 엔터테인먼트, 소셜 네트워크 등에서 널리 사용됩니다."
    },
    {
        "id": 15,
        "title": "지식 그래프와 온톨로지",
        "content": "지식 그래프는 실세계 개체와 그들 간의 관계를 그래프 형태로 표현한 지식베이스입니다. 온톨로지와 함께 의미론적 정보를 구조화하며, 추론과 질의응답 시스템의 기반이 됩니다. 검색 엔진, 가상 비서, 추천 시스템 등에서 활용되며, 기계학습 모델의 배경 지식으로도 사용됩니다."
    }
]

# 문서 데이터프레임 생성
df_documents = pd.DataFrame(sample_documents)
print(f"총 {len(df_documents)}개의 문서가 로드되었습니다.")
df_documents[['id', 'title']]

총 15개의 문서가 로드되었습니다.


,id,title
0,1,파이썬 기초 프로그래밍
1,2,머신러닝 입문
2,3,딥러닝의 이해
3,4,자연어 처리 기술
4,5,데이터 시각화 방법론
5,6,인공신경망의 기초
6,7,컴퓨터 비전 개요
7,8,강화학습 알고리즘
8,9,빅데이터 처리 기술
9,10,자연어 이해(NLU) 기술


### 2.1. 간단한 키워드 기반 검색 구현

먼저 간단한 키워드 기반 검색 함수를 구현해보겠습니다.

In [36]:
def keyword_search(query, documents, n=5):
    """
    간단한 키워드 기반 검색 함수
    query: 검색어
    documents: 검색할 문서 데이터프레임
    n: 반환할 최대 결과 수
    """
    # 검색어를 소문자로 변환하고 단어로 분리
    query_words = query.lower().split()
    
    scores = []
    
    # 각 문서에 대해 단어 일치 횟수 계산
    for i, row in documents.iterrows():
        title = str(row["title"]).lower()
        content = str(row["content"]).lower()
        score = sum(1 for word in query_words if word in title or word in content)
        if score > 0:  # 점수가 0보다 큰 경우만 추가
            scores.append((score, i))  # 인덱스를 저장
    
    # 점수 기준 내림차순 정렬
    scores = sorted(scores, reverse=True)
    
    # 상위 n개 결과 반환
    results = []
    for score, idx in scores[:n]:
        results.append(documents.iloc[idx])
    
    return results

# 검색 테스트
query = "파이썬 프로그래밍 기초"
results = keyword_search(query, df_documents)

print(f"쿼리: '{query}'에 대한 검색 결과:")
for i, result in enumerate(results):
    print(f"\n결과 {i+1}:")
    print(f"제목: {result['title']}")
    print(f"내용: {result['content'][:100]}...")

쿼리: '파이썬 프로그래밍 기초'에 대한 검색 결과:

결과 1:
제목: 파이썬 기초 프로그래밍
내용: 파이썬은 초보자부터 전문가까지 널리 사용되는 프로그래밍 언어입니다. 간결한 문법과 다양한 라이브러리로 인해 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 활용됩니다. 변수,...

결과 2:
제목: 인공신경망의 기초
내용: 인공신경망은 인간 뇌의 뉴런 구조에서 영감을 받은 컴퓨팅 모델입니다. 입력층, 은닉층, 출력층으로 구성되며 각 노드는 가중치와 활성화 함수를 통해 연결됩니다. 역전파 알고리즘을 통...


## 3. 기본 RAG 파이프라인 구현

이제 검색 결과를 프롬프트에 통합하여 응답을 생성하는 기본 RAG 파이프라인을 구현해보겠습니다.

In [37]:
def rag_pipeline(query, documents, search_func=keyword_search, n_docs=2, verbose=True):
    """
    기본 RAG 파이프라인 함수
    
    Args:
        query (str): 사용자 질문
        documents (DataFrame): 문서 데이터프레임
        search_func (function): 검색 함수
        n_docs (int): 검색할 문서 수
        verbose (bool): 중간 결과 출력 여부
        
    Returns:
        str: 생성된 응답
    """
    # 1. 검색
    retrieved_docs = search_func(query, documents, n=n_docs)
    
    # 2. 검색된 문서 출력 (verbose=True인 경우)
    if verbose:
        print(f"검색된 문서 ({len(retrieved_docs)}개):")
        for i, doc in enumerate(retrieved_docs):
            print(f"\n문서 {i+1}:")
            print(f"제목: {doc['title']}")
            print(f"내용: {doc['content'][:150]}...")
    
    # 3. 컨텍스트 구성
    context = "\n\n".join([f"제목: {doc['title']}\n내용: {doc['content']}" for doc in retrieved_docs])
    
    # 4. 프롬프트 구성
    prompt = f"""
    다음은 사용자의 질문과 관련된 문서들입니다:
    
    {context}
    
    위 문서들의 정보를 바탕으로 다음 질문에 답변해주세요:
    {query}
    
    위 문서의 내용에 없는 정보는 포함하지 말고, 문서 내용을 기반으로만 답변해주세요.
    답변에 사용한 정보의 출처(문서 제목)를 마지막에 표시해주세요.
    """
    
    # 5. 응답 생성
    response = get_completion(prompt)
    
    return response

# RAG 파이프라인 테스트
user_question = "파이썬은 어떤 분야에서 활용될 수 있나요?"
response = rag_pipeline(user_question, df_documents)

print(f"\n질문: {user_question}")
print("\nRAG 응답:")
print(response)

검색된 문서 (2개):

문서 1:
제목: 파이썬 기초 프로그래밍
내용: 파이썬은 초보자부터 전문가까지 널리 사용되는 프로그래밍 언어입니다. 간결한 문법과 다양한 라이브러리로 인해 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 활용됩니다. 변수, 자료형, 조건문, 반복문 등 기본 개념을 이해하면 쉽게 배울 수 있습니다....

문서 2:
제목: 컴퓨터 비전 개요
내용: 컴퓨터 비전은 기계가 디지털 이미지와 비디오를 분석하고 이해할 수 있게 하는 AI 분야입니다. 이미지 분류, 객체 감지, 이미지 분할, 얼굴 인식 등의 작업을 수행합니다. CNN(합성곱 신경망)이 주로 사용되며, 자율주행차, 의료영상 분석 등에 활용됩니다....

질문: 파이썬은 어떤 분야에서 활용될 수 있나요?

RAG 응답:
파이썬은 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 활용될 수 있습니다. (출처: 파이썬 기초 프로그래밍)

토큰 수: 60


## 4. RAG vs. 일반 프롬프팅 비교

RAG 접근 방식과 일반 프롬프팅의 응답 품질을 비교해보겠습니다.

In [38]:
# RAG와 일반 프롬프팅 비교
test_question = "딥러닝과 머신러닝의 차이점은 무엇인가요?"

# RAG 응답
rag_response = rag_pipeline(test_question, df_documents, n_docs=2)

# 일반 프롬프팅 응답
regular_prompt = f"""
다음 질문에 답변해주세요: {test_question}
"""
regular_response = get_completion(regular_prompt)

print("===== 질문 =====")
print(test_question)
print("\n===== RAG 응답 =====")
print(rag_response)
print("\n===== 일반 프롬프팅 응답 =====")
print(regular_response)

검색된 문서 (1개):

문서 1:
제목: 딥러닝의 이해
내용: 딥러닝은 인공신경망을 여러 층으로 쌓아 복잡한 패턴을 학습하는 머신러닝의 한 분야입니다. 이미지 인식, 자연어 처리, 음성 인식 등에서 뛰어난 성능을 보이며, CNN, RNN, Transformer 등 다양한 아키텍처가 있습니다. 대량의 데이터와 GPU를 활용한 학습이...
===== 질문 =====
딥러닝과 머신러닝의 차이점은 무엇인가요?

===== RAG 응답 =====
딥러닝은 머신러닝의 한 분야로, 인공신경망을 여러 층으로 쌓아 복잡한 패턴을 학습합니다. 따라서, 딥러닝은 머신러닝의 특정한 접근 방식 중 하나로 볼 수 있습니다. 머신러닝은 데이터를 통해 학습하는 일반적인 알고리즘을 포함하며, 딥러닝은 이러한 알고리즘 중 인공신경망을 깊게 쌓아 학습하는 방법을 사용하는 것입니다. 

출처: 딥러닝의 이해

===== 일반 프롬프팅 응답 =====
딥러닝과 머신러닝은 모두 인공지능의 하위 분야이지만, 몇 가지 중요한 차이점이 있습니다.

1. **구조**:
   - **머신러닝**: 머신러닝은 데이터에서 패턴을 학습하고 예측을 수행하는 알고리즘의 집합입니다. 머신러닝 알고리즘에는 선형 회귀, 로지스틱 회귀, 결정 트리, 서포트 벡터 머신(SVM), K-평균 클러스터링 등이 포함됩니다.
   - **딥러닝**: 딥러닝은 인공신경망을 사용하여 데이터를 처리하고 학습하는 머신러닝의 하위 분야입니다. 특히, 여러 층의 신경망을 사용하여 복잡한 패턴을 학습하는 방법을 강조합니다. 딥러닝 모델에는 CNN(Convolutional Neural Networks), RNN(Recurrent Neural Networks), 트랜스포머(Transformers) 등이 있습니다.

2. **데이터 처리**:
   - **머신러닝**: 머신러닝 모델은 일반적으로 구조화된 데이터를 필요로 하며, 특징 공학(feature engineering)이 중요합니다. 데이터의 특징을 사람이 직접 정의하고 선택해야 하는 경우가 많

## 5. 개선된 RAG: 벡터 검색 구현

키워드 기반 검색보다 더 고급 검색 방법인 벡터 검색(임베딩 기반 검색)을 구현해보겠습니다. 벡터 검색은 단어의 의미적 유사성을 고려하여 더 관련성 높은 문서를 찾을 수 있습니다.

In [47]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def vector_search(query, documents, n=5):
    """
    사전학습된 변환기 모델을 사용한 벡터 검색 함수
    
    Args:
        query (str): 검색어
        documents (DataFrame): 검색할 문서 데이터프레임
        n (int): 반환할 최대 결과 수
        
    Returns:
        list: 검색 결과 문서 리스트
    """
    # 한국어 지원 모델 로드
    model = SentenceTransformer('jhgan/ko-sbert-nli')
    
    # 문서 본문과 제목을 합쳐서 텍스트 데이터 준비
    corpus = [(doc['title'] + ' ' + doc['content']) for _, doc in documents.iterrows()]
    
    # 문서 임베딩 생성
    corpus_embeddings = model.encode(corpus)
    
    # 쿼리 임베딩 생성
    query_embedding = model.encode(query)
    
    # 코사인 유사도 계산
    similarities = cosine_similarity([query_embedding], corpus_embeddings)[0]
    
    # 유사도 기준 내림차순 정렬
    top_indices = similarities.argsort()[::-1][:n]
    
    results = []
    for idx in top_indices:
        if similarities[idx] > 0:
            results.append(documents.iloc[idx])
    
    return results

# 벡터 검색 테스트
query = "인공지능과 머신러닝"
results = vector_search(query, df_documents)

print(f"쿼리: '{query}'에 대한 벡터 검색 결과:")
for i, result in enumerate(results):
    print(f"\n결과 {i+1} (유사도 점수 높은 순):")
    print(f"제목: {result['title']}")
    print(f"내용: {result['content'][:100]}...")

/Users/jaden/Projects/prompting-tutorials/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


쿼리: '인공지능과 머신러닝'에 대한 벡터 검색 결과:

결과 1 (유사도 점수 높은 순):
제목: 딥러닝의 이해
내용: 딥러닝은 인공신경망을 여러 층으로 쌓아 복잡한 패턴을 학습하는 머신러닝의 한 분야입니다. 이미지 인식, 자연어 처리, 음성 인식 등에서 뛰어난 성능을 보이며, CNN, RNN, T...

결과 2 (유사도 점수 높은 순):
제목: 머신러닝 입문
내용: 머신러닝은 컴퓨터가 데이터로부터 패턴을 학습하고 예측할 수 있게 하는 기술입니다. 지도학습, 비지도학습, 강화학습 등 다양한 학습 방식이 있으며, 분류, 회귀, 클러스터링 등의 작...

결과 3 (유사도 점수 높은 순):
제목: 컴퓨터 비전 개요
내용: 컴퓨터 비전은 기계가 디지털 이미지와 비디오를 분석하고 이해할 수 있게 하는 AI 분야입니다. 이미지 분류, 객체 감지, 이미지 분할, 얼굴 인식 등의 작업을 수행합니다. CNN(...

결과 4 (유사도 점수 높은 순):
제목: 추천 시스템 알고리즘
내용: 추천 시스템은 사용자의 선호도와 행동에 기반하여 개인화된 항목을 제안하는 AI 응용 프로그램입니다. 협업 필터링, 콘텐츠 기반 필터링, 하이브리드 방식 등의 접근법이 있으며, 행렬...

결과 5 (유사도 점수 높은 순):
제목: 강화학습 알고리즘
내용: 강화학습은 에이전트가 환경과 상호작용하며 보상을 최대화하는 방향으로 행동을 학습하는 기계학습 방법입니다. Q-러닝, 정책 경사법, 딥 Q-네트워크(DQN) 등의 알고리즘이 있으며,...


## 6. 키워드 검색과 벡터 검색 비교

이제 키워드 검색과 벡터 검색의 성능을 비교해보겠습니다.

In [50]:
def compare_search_methods(query):
    """
    키워드 검색과 벡터 검색 결과 비교 함수
    
    Args:
        query (str): 검색어
    """
    print(f"검색어: '{query}'")
    print("\n=== 키워드 검색 결과 ===")
    keyword_results = keyword_search(query, df_documents, n=5)
    for i, result in enumerate(keyword_results):
        print(f"\n결과 {i+1}:")
        print(f"제목: {result['title']}")
        print(f"내용: {result['content'][:100]}...")
    
    print("\n=== 벡터 검색 결과 ===")
    vector_results = vector_search(query, df_documents, n=5)
    for i, result in enumerate(vector_results):
        print(f"\n결과 {i+1}:")
        print(f"제목: {result['title']}")
        print(f"내용: {result['content'][:100]}...")

# 검색 방법 비교 테스트
test_query = "신경망 기술"
compare_search_methods(test_query)

검색어: '신경망 기술'

=== 키워드 검색 결과 ===

결과 1:
제목: 추천 시스템 알고리즘
내용: 추천 시스템은 사용자의 선호도와 행동에 기반하여 개인화된 항목을 제안하는 AI 응용 프로그램입니다. 협업 필터링, 콘텐츠 기반 필터링, 하이브리드 방식 등의 접근법이 있으며, 행렬...

결과 2:
제목: 신경망 최적화 기법
내용: 신경망 최적화는 모델의 성능을 향상시키기 위한 다양한 기법을 포함합니다. 경사하강법, 모멘텀, Adam 등의 최적화 알고리즘과 배치 정규화, 드롭아웃, 조기 종료 등의 정규화 기법...

결과 3:
제목: 인공지능 윤리
내용: 인공지능 윤리는 AI 시스템 개발과 사용에 관련된 도덕적 문제를 다룹니다. 프라이버시, 편향성, 투명성, 책임성, 자율성 등이 주요 이슈이며, 인공지능의 발전에 따라 사회적, 법적...

결과 4:
제목: 자연어 이해(NLU) 기술
내용: 자연어 이해는 컴퓨터가 인간 언어의 의미를 파악하는 NLP의 하위 분야입니다. 의도 분류, 개체명 인식, 감정 분석 등을 통해 텍스트의 의미를 이해합니다. BERT, GPT 등의 ...

결과 5:
제목: 빅데이터 처리 기술
내용: 빅데이터 처리는 대용량 데이터를 효율적으로 저장, 처리, 분석하는 기술입니다. Hadoop, Spark, Kafka 등의 프레임워크가 활용되며, 분산 처리를 통해 대규모 데이터를 ...

=== 벡터 검색 결과 ===

결과 1:
제목: 인공신경망의 기초
내용: 인공신경망은 인간 뇌의 뉴런 구조에서 영감을 받은 컴퓨팅 모델입니다. 입력층, 은닉층, 출력층으로 구성되며 각 노드는 가중치와 활성화 함수를 통해 연결됩니다. 역전파 알고리즘을 통...

결과 2:
제목: 컴퓨터 비전 개요
내용: 컴퓨터 비전은 기계가 디지털 이미지와 비디오를 분석하고 이해할 수 있게 하는 AI 분야입니다. 이미지 분류, 객체 감지, 이미지 분할, 얼굴 인식 등의 작업을 수행합니다. CNN(...

결과 3:
제목: 자연어 이해(NLU) 기술
내용: 자연어 이해는 컴


## 7. 벡터 검색 기반 RAG 파이프라인

마지막으로 벡터 검색을 활용한 RAG 파이프라인을 구현하고 테스트해보겠습니다.

In [51]:
# 벡터 검색 기반 RAG 테스트
vector_rag_question = "인공지능 기술의 종류는 무엇인가요?"
print("\n===== 벡터 검색 기반 RAG =====")
print(f"질문: {vector_rag_question}")
vector_rag_response = rag_pipeline(vector_rag_question, df_documents, search_func=vector_search)
print("\nRAG 응답:")
print(vector_rag_response)

# 키워드 검색 기반 RAG와 비교
print("\n===== 키워드 검색 기반 RAG =====")
print(f"질문: {vector_rag_question}")
keyword_rag_response = rag_pipeline(vector_rag_question, df_documents, search_func=keyword_search)
print("\nRAG 응답:")
print(keyword_rag_response)


===== 벡터 검색 기반 RAG =====
질문: 인공지능 기술의 종류는 무엇인가요?
검색된 문서 (2개):

문서 1:
제목: 컴퓨터 비전 개요
내용: 컴퓨터 비전은 기계가 디지털 이미지와 비디오를 분석하고 이해할 수 있게 하는 AI 분야입니다. 이미지 분류, 객체 감지, 이미지 분할, 얼굴 인식 등의 작업을 수행합니다. CNN(합성곱 신경망)이 주로 사용되며, 자율주행차, 의료영상 분석 등에 활용됩니다....

문서 2:
제목: 추천 시스템 알고리즘
내용: 추천 시스템은 사용자의 선호도와 행동에 기반하여 개인화된 항목을 제안하는 AI 응용 프로그램입니다. 협업 필터링, 콘텐츠 기반 필터링, 하이브리드 방식 등의 접근법이 있으며, 행렬 분해, 딥러닝, 강화학습 등 다양한 기술이 활용됩니다. 전자상거래, 엔터테인먼트, 소셜 ...

RAG 응답:
인공지능 기술의 종류로는 컴퓨터 비전과 추천 시스템이 있습니다. 컴퓨터 비전은 기계가 디지털 이미지와 비디오를 분석하고 이해할 수 있게 하며, 이미지 분류, 객체 감지, 이미지 분할, 얼굴 인식 등의 작업을 수행합니다. 추천 시스템은 사용자의 선호도와 행동에 기반하여 개인화된 항목을 제안하는 응용 프로그램으로, 협업 필터링, 콘텐츠 기반 필터링, 하이브리드 방식 등의 접근법이 사용됩니다. 

출처: 컴퓨터 비전 개요, 추천 시스템 알고리즘

토큰 수: 232

===== 키워드 검색 기반 RAG =====
질문: 인공지능 기술의 종류는 무엇인가요?
검색된 문서 (2개):

문서 1:
제목: 인공지능 윤리
내용: 인공지능 윤리는 AI 시스템 개발과 사용에 관련된 도덕적 문제를 다룹니다. 프라이버시, 편향성, 투명성, 책임성, 자율성 등이 주요 이슈이며, 인공지능의 발전에 따라 사회적, 법적 규제 프레임워크 개발이 중요해지고 있습니다. AI 기술이 공정하고 안전하게 사용되도록 하...

문서 2:
제목: 파이썬 기초 프로그래밍
내용: 파이썬은 초보자부터 전문가까지 널리 사용되는 프로그래밍 언어입니다. 간결한 문법과 다

## 8. RAG 성능 개선 방법

RAG 시스템의 성능을 더욱 향상시키는 방법에는 여러 가지가 있습니다.


In [32]:
# RAG 성능 개선을 위한 방법
print("===== RAG 성능 개선 방법 =====")
improvement_prompt = """
다음의 RAG(검색 증강 생성) 시스템 성능 개선 방법에 대해 설명해주세요:
1. 더 좋은 임베딩 모델 사용하기
2. 청킹(chunking) 전략 최적화
3. 재순위화(reranking) 적용하기
4. 쿼리 확장 및 개선
"""
improvement_response = get_completion(improvement_prompt)
print(improvement_response)

===== RAG 성능 개선 방법 =====
RAG(검색 증강 생성) 시스템의 성능을 개선하기 위한 다양한 전략들이 존재하며, 각각의 접근 방식은 정보 검색 및 생성의 정확성과 효율성을 높이는 데 중점을 둡니다. 아래에 각 방법에 대한 설명을 제공합니다.

1. **더 좋은 임베딩 모델 사용하기**:
   - 임베딩 모델은 텍스트 데이터를 벡터로 변환하여 기계 학습 알고리즘이 이해할 수 있도록 합니다. 더 나은 임베딩 모델을 사용하는 것은 검색 및 생성 시스템의 성능을 향상시키는 데 중요한 역할을 합니다. 최신의 강력한 임베딩 모델(예: BERT, RoBERTa, Sentence Transformers 등)은 문맥을 보다 잘 이해하고 유사성을 더 정확하게 계산할 수 있어, 관련성이 높은 결과를 제공할 수 있습니다.

2. **청킹(chunking) 전략 최적화**:
   - 청킹은 문서를 작은 청크로 나누는 과정으로, 각 청크가 검색 시스템에 의해 개별적으로 처리될 수 있습니다. 최적화된 청킹 전략은 각 청크가 의미 있는 정보를 포함하도록 하여 검색의 품질을 높일 수 있습니다. 청킹의 크기와 경계를 조절함으로써 검색의 정밀도와 재현율을 개선할 수 있습니다. 또한, 적절한 청킹은 시스템의 메모리 사용을 줄이고 처리 속도를 높일 수 있습니다.

3. **재순위화(reranking) 적용하기**:
   - 재순위화는 초기 검색 결과에서 더 높은 품질의 결과를 선별하여 상위에 배치하는 과정입니다. 초기 검색 결과에서 관련성이 높은 문서를 식별하고, 사용자의 쿼리에 대한 문맥을 더 잘 반영하는 결과를 상위로 배치함으로써 검색의 정확성을 높일 수 있습니다. 이를 위해 추가적인 모델(예: 더 정교한 랭킹 모델)을 사용할 수 있습니다.

4. **쿼리 확장 및 개선**:
   - 쿼리 확장은 사용자의 원래 쿼리에 추가적인 관련 단어나 구문을 포함시켜 검색의 범위와 정확성을 높이는 방법입니다. 동의어, 유의어, 관련 키워드 등을 추가하여 보다 포괄적인 검색을 수행할 수 있습니